# Day 5 Practice: Validation Architecture & Baselines

**Role:** Competitive Kaggle Grandmaster

Validation is the difference between a Gold Medal and a 'shake-down' on the private leaderboard. Today, you will build a robust OOF framework, create a baseline, and learn to hunt down the most dangerous silent killer: Data Leakage.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import sys
import os

# Setup validation
try:
    import ipytest
except ImportError:
    !pip install -q ipytest
    import ipytest

if not os.path.exists('validate_day5.py'):
    print('⚠️ validate_day5.py not found!')
else:
    import validate_day5

ipytest.autoconfig()


## 1. Setup Dataset
Generating an imbalanced classification dataset with missing values.


In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, weights=[0.9, 0.1], random_state=42)
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(20)])
df['Target'] = y
df.iloc[0:100, 0] = np.nan # Introduce missing values
df_test = df.sample(500).drop(columns=['Target'])
df_test['ID'] = range(500)


## Challenge 1: Robust Stratified 5-Fold CV Loop
**Task:** Implement a StratifiedKFold loop. Track OOF predictions in an array `oof_preds` (size of training set).


In [ ]:
# YOUR CODE HERE
# 1. Init oof_preds array with zeros
# 2. Setup StratifiedKFold(n_splits=5)
# 3. Inside loop: train model, log predictions for the current val_fold


In [ ]:
%%ipytest -qq
try:
    validate_day5.test_task_1()
except NameError: pass


## Challenge 2: Baseline Model & Submission
**Task:** Train a `RandomForestClassifier` inside your fold loop. After completing the loop, generate a `submission` DataFrame with columns `ID` and `Prediction` (averaging the fold probabilities).


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day5.test_task_2()
except NameError: pass


## Challenge 3: Introducing Data Leakage
**Task:** Intentionally introduce a 'leaked' pipeline. 
1. Globally impute and scale the *entire dataset* before splitting into folds. 
2. Train your baseline. 
3. Compare the score to your robust model and store in `leaked_score`.


In [ ]:
# YOUR CODE HERE
# 1. Global Impute/Scale
# 2. Run CV loop with leaked data
# 3. Store result in leaked_score


In [ ]:
%%ipytest -qq
try:
    validate_day5.test_task_3()
except NameError: pass


## Challenge 4: Refactoring for Robustness
**Task:** Refactor your pipeline from Challenge 3. Ensure `SimpleImputer` and `StandardScaler` are fitted **strictly on the training fold** and transformed on the validation fold inside the loop. Calculate the score and store in `robust_score`.


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day5.test_task_4()
except NameError: pass


<details>
<summary style='color: #d35400; font-weight: bold; cursor: pointer;'>🏆 Click for Grandmaster Optimized Solutions</summary>

```python
# 1 & 2: CV Loop with RF
oof_preds = np.zeros(len(df))
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
models = []
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['Target'])):
    X_tr, X_val = df.iloc[train_idx].drop(columns='Target'), df.iloc[val_idx].drop(columns='Target')
    y_tr, y_val = df.iloc[train_idx]['Target'], df.iloc[val_idx]['Target']
    model = RandomForestClassifier(n_estimators=50).fit(X_tr.fillna(0), y_tr)
    oof_preds[val_idx] = model.predict_proba(X_val.fillna(0))[:, 1]
    models.append(model)

# 3. Leakage Bug: Global Scaling
df_leaked = df.copy()
df_leaked.iloc[:, :-1] = StandardScaler().fit_transform(df_leaked.fillna(0).iloc[:, :-1])
# ... run CV on df_leaked ...

# 4. Leakage-Free Refactor
for train_idx, val_idx in skf.split(df, df['Target']):
    X_tr, X_val = df.iloc[train_idx].drop(columns='Target'), df.iloc[val_idx].drop(columns='Target')
    imputer = SimpleImputer().fit(X_tr)
    X_tr_proc = imputer.transform(X_tr)
    X_val_proc = imputer.transform(X_val)
    # ... fit model on X_tr_proc ...
```
</details>
